# 02 - Let's design a clustered A/B test

**Business Loves Uplift: Causal Impact Evaluation With Python**<br>
ODSC AI West 2026 · Alex Molak · [CausalPython.io](https://causalpython.io)

<a href="https://colab.research.google.com/github/AlxndrMlk/talks-and-workshops/blob/main/2026-10--odsc-ai/02%20-%20Clustered%20AB%20test.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>


**The story.** A retailer's field sales reps each look after 120-200 stores. The data team built a recommender that tells a rep which new products (SKUs) to list in each store. How much does it lift store sales?

The data is synthetic, so we know the true answer. Weekly sales are in k$ per store.

The rep carries the treatment, so the rep is the unit we randomize: all stores of a rep get the same arm. In this notebook we see what this costs in power and how to analyze such a test.

In [ ]:
# Colab only: install the packages and get the data generator (locally, use the workshop env)
import sys
if "google.colab" in sys.modules:
    !pip install -q dowhy==0.14 econml==0.17.0 cluster-experiments==0.31.0 mlsynth==1.0.0
    !wget -q -nc https://raw.githubusercontent.com/AlxndrMlk/talks-and-workshops/main/2026-10--odsc-ai/uplift_data.py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from cluster_experiments import NormalPowerAnalysis, OLSAnalysis, ClusteredOLSAnalysis

import uplift_data as ud

BLUE, GOLD, GREY, RED, DARK = "#0057d9", "#c8860d", "#bbbbbb", "#c0392b", "#222222"
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
d = ud.make_datasets()     # seed 23: everyone gets the same world
truth = d["truth"]         # the answer key; we only peek at it at the end of each section
print(ud.fingerprint(d))   # should print 13302.553/5496.520

## How alike are a rep's stores?

The intra-cluster correlation (ICC) is the share of the variance in sales that comes from differences between reps. Even a small ICC matters when clusters are big: the *design effect* `1 + (m - 1) * ICC` tells how much the variance grows compared to randomizing stores, where `m` is the number of stores per rep.

In [ ]:
history = d["history"]
stores = history.groupby(["store_id", "rep_id"], as_index=False).sales.mean()


def icc(df, y, group):
    """One-way ANOVA estimate of the intra-cluster correlation."""
    g = df.groupby(group)[y]
    k = g.size().mean()
    msb, msw = g.mean().var() * k, g.var().mean()
    return (msb - msw) / (msb + (k - 1) * msw)


rho = icc(stores, "sales", "rep_id")
m = stores.groupby("rep_id").size().mean()
deff = 1 + (m - 1) * rho
print(f"ICC = {rho:.3f}, stores per rep = {m:.0f}, design effect = {deff:.1f}")
print(f"40 reps x {m:.0f} stores = {40 * m:.0f} stores, worth about {40 * m / deff:.0f} independent stores")

## Power: how many reps do we need?

Same simulation as before, but now the splitter assigns whole reps to an arm, and the analysis clusters the standard errors by rep. We compare it with what the store-level analysis would claim on the same stores.

In [ ]:
config = {"target_col": "sales", "n_simulations": 50, "seed": 0}
pw_rep = NormalPowerAnalysis.from_dict({
    "splitter": "clustered", "analysis": "clustered_ols", "cluster_cols": ["rep_id"], **config})
pw_store = NormalPowerAnalysis.from_dict({
    "splitter": "non_clustered", "analysis": "ols", "cluster_cols": ["store_id"], **config})

rng = np.random.default_rng(0)
rows = []
for n_reps in [20, 40, 80, 120, 160]:
    reps = rng.choice(stores.rep_id.unique(), n_reps, replace=False)
    sample = stores[stores.rep_id.isin(reps)]
    mean = sample.sales.mean()
    rows.append({"n_reps": n_reps,
                 "MDE, randomize reps": pw_rep.mde(sample, power=0.8) / mean,
                 "MDE, store-level analysis": pw_store.mde(sample, power=0.8) / mean})
mdes = pd.DataFrame(rows).set_index("n_reps")
mdes.style.format("{:.1%}")

In [ ]:
ax = mdes.plot(marker="o", color=[BLUE, GREY])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("number of reps in the test")
ax.set_ylabel("MDE (% of sales), 80% power")
plt.show()

With 40 reps the clustered test can detect an effect of about 9% of sales. That's borderline if we expect around 10%, and nowhere near the 2% the store-level analysis promised. 80 reps would bring it to about 5%.

## Analyze the rep-randomized test

`ab_rep` holds the same 40 reps, now with whole reps randomized.

In [ ]:
ab = d["ab_rep"]
print(ab.groupby("treated").agg(reps=("rep_id", "nunique"), stores=("store_id", "size")))
ab

We analyze it with cluster-experiments too: `OLSAnalysis` with regular standard errors, and `ClusteredOLSAnalysis`, which clusters them by rep. Both compare sales in the test weeks between the arms.

In [ ]:
setup = {"target_col": "sales_test", "treatment_col": "treated", "treatment": 1}
naive = OLSAnalysis(cov_type="nonrobust", **setup).get_inference_results(ab, alpha=0.05)
clustered = ClusteredOLSAnalysis(cluster_cols=["rep_id"], **setup).get_inference_results(ab, alpha=0.05)

base = ab.loc[ab.treated == 0, "sales_test"].mean()
for name, r in [("regular SEs", naive), ("clustered by rep", clustered)]:
    width = r.conf_int.upper - r.conf_int.lower
    print(f"{name:17s} effect {r.ate:.2f} k$ [{r.conf_int.lower:.2f}, {r.conf_int.upper:.2f}], "
          f"interval width {width:.2f} k$, {r.ate / base:.1%} of control sales")

ratio = (clustered.conf_int.upper - clustered.conf_int.lower) / (naive.conf_int.upper - naive.conf_int.lower)
print(f"\nthe clustered interval is {ratio:.1f}x wider; sqrt(design effect) = {np.sqrt(deff):.1f}")

Same estimate, but the clustered interval is more than 4x wider. That's the design effect from above: variance 19 times larger, so the interval is about √19 ≈ 4.4 times wider. The clustered interval is the honest one.

**The same analysis in plain statsmodels.** Under the hood, cluster-experiments runs a statsmodels regression, so you can get the same numbers yourself:

In [ ]:
formula = "sales_test ~ treated"
for name, f in [("regular SEs", smf.ols(formula, ab).fit()),
                ("clustered by rep", smf.ols(formula, ab).fit(cov_type="cluster", cov_kwds={"groups": ab.rep_id}))]:
    lo, hi = f.conf_int().loc["treated"]
    print(f"{name:17s} effect {f.params['treated']:.2f} k$ [{lo:.2f}, {hi:.2f}]")

So why use the package? It covers the whole test in one place: the splitter that randomizes reps, the power and MDE calculation, and the analysis, all built on the same settings. The analysis you plan the power for is the analysis you run at the end. It also comes with extras you'd otherwise write yourself: CUPED/CUPAC variance reduction, switchback and washover designs, the delta method for ratio metrics, and power simulations for any of its analyses. Plain statsmodels is fine for a one-off regression; the package pays off when you design and run tests regularly.

## Answer key

In [ ]:
est, lo, hi = clustered.ate, clustered.conf_int.lower, clustered.conf_int.upper
print(f"true effect on treated stores:   {truth['ab_rep_direct_abs']:.2f} k$")
print(f"effect leaked onto controls:     {truth['ab_rep_control_spill_abs']:.2f} k$  (only via reps in the same region)")
print(f"what the test should measure:    {truth['ab_rep_expected_diff_abs']:.2f} k$ = {truth['ab_rep_expected_diff_pct']:.1%}")
print(f"our estimate:                    {est:.2f} k$ [{lo:.2f}, {hi:.2f}] = {est / base:.1%}")

Randomizing reps fixed both problems: almost nothing leaks onto the control arm, and the clustered interval covers the truth.

## Extra: CUPED

The clustered interval is honest, but wide. We can narrow it with CUPED: adjust for `sales_pre`, the sales before the test. The recipe:

1. `theta = cov(sales_pre, sales_test) / var(sales_pre)`
2. Adjusted outcome: `sales_test - theta * (sales_pre - mean(sales_pre))`
3. Compare the arms on the adjusted outcome.

Adding `sales_pre` as a covariate in the regression does the same thing in one step.

In [ ]:
theta = np.cov(ab.sales_pre, ab.sales_test)[0, 1] / ab.sales_pre.var()
ab_cuped = ab.assign(sales_cuped=ab.sales_test - theta * (ab.sales_pre - ab.sales_pre.mean()))

for name, target, covariates in [("no adjustment", "sales_test", None),
                                 ("CUPED", "sales_cuped", None),
                                 ("regression on sales_pre", "sales_test", ["sales_pre"])]:
    r = ClusteredOLSAnalysis(cluster_cols=["rep_id"], target_col=target, treatment_col="treated", treatment=1,
                             covariates=covariates).get_inference_results(ab_cuped, alpha=0.05)
    print(f"{name:24s} effect {r.ate:.2f} k$ [{r.conf_int.lower:.2f}, {r.conf_int.upper:.2f}]")

CUPED and the regression give the same answer, with an interval almost three times narrower than without the adjustment: pre-test sales explain most of the differences between stores. The only rule: the covariate must be measured before the test starts, so the treatment can't affect it.

## What comes next

The test ran on 40 reps for 8 weeks. Then the business made its call: the recommender went live for **every rep in the North** at once, in week 76. No coin flips this time, so from here on there's no randomization to lean on.

Did the rollout work, and by how much? Next: double machine learning (notebook 03) and synthetic control (notebook 04), on the same rollout data.